# 01 — Exploratory Data Analysis

This notebook performs an in-depth exploration of the Lending Club Loan dataset (2007–2018).  
We examine distributions, class balance, missing values, correlations, and univariate relationships with the target variable (`bad_loan`).

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 50)
%matplotlib inline

## 1.1 Load Raw Data

In [ ]:
df = pd.read_csv(config.RAW_CSV, low_memory=False)
print(f"Shape: {df.shape}")
df.head()

## 1.2 Target Variable – Loan Status Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All statuses
status_counts = df["loan_status"].value_counts()
status_counts.plot.barh(ax=axes[0], color=sns.color_palette("viridis", len(status_counts)))
axes[0].set_title("Loan Status Distribution (All)")
axes[0].set_xlabel("Count")

# Binary target
binary = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]["loan_status"]
binary.value_counts().plot.bar(ax=axes[1], color=["#2ecc71", "#e74c3c"])
axes[1].set_title("Binary Target (Fully Paid vs Charged Off)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"\nDefault rate: {(binary == 'Charged Off').mean():.2%}")

## 1.3 Missing Values Analysis

In [ ]:
missing = df[config.KEY_COLUMNS].isnull().mean().sort_values(ascending=False)
missing_pct = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, 6))
missing_pct.plot.barh(ax=ax, color="#e67e22")
ax.axvline(x=0.4, color="red", linestyle="--", label="40% threshold")
ax.set_title("Missing Value Percentage – Key Columns")
ax.set_xlabel("Fraction Missing")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Columns above 40% missing: {list(missing_pct[missing_pct > 0.4].index)}")

## 1.4 Numeric Feature Distributions

In [ ]:
# Subset to binary target only
df_binary = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
df_binary["bad_loan"] = (df_binary["loan_status"] == "Charged Off").astype(int)

num_features = ["loan_amnt", "int_rate", "annual_inc", "dti", "revol_util",
                "fico_range_low", "installment", "revol_bal"]
# Filter to columns that exist
num_features = [c for c in num_features if c in df_binary.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, col in zip(axes.ravel(), num_features):
    # Convert if needed
    series = pd.to_numeric(df_binary[col].astype(str).str.replace("%", ""), errors="coerce")
    series.hist(ax=ax, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_ylabel("")

plt.suptitle("Numeric Feature Distributions", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 1.5 Default Rate by Categorical Features

In [ ]:
cat_features = ["grade", "home_ownership", "verification_status", "term"]
cat_features = [c for c in cat_features if c in df_binary.columns]

fig, axes = plt.subplots(1, len(cat_features), figsize=(6 * len(cat_features), 5))
if len(cat_features) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_features):
    rates = df_binary.groupby(col)["bad_loan"].mean().sort_values(ascending=False)
    rates.plot.bar(ax=ax, color=sns.color_palette("RdYlGn_r", len(rates)))
    ax.set_title(f"Default Rate by {col}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Default Rate")
    ax.set_ylim(0, rates.max() * 1.3)
    for i, v in enumerate(rates):
        ax.text(i, v + 0.005, f"{v:.1%}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 1.6 Correlation Heatmap

In [ ]:
corr_cols = ["loan_amnt", "funded_amnt", "installment", "annual_inc",
             "dti", "fico_range_low", "fico_range_high", "open_acc",
             "pub_rec", "revol_bal", "total_acc", "bad_loan"]
corr_cols = [c for c in corr_cols if c in df_binary.columns]

corr_matrix = df_binary[corr_cols].apply(pd.to_numeric, errors="coerce").corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax, linewidths=0.5)
ax.set_title("Correlation Heatmap – Key Numeric Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 1.7 FICO Score vs Default Rate

In [ ]:
if "fico_range_low" in df_binary.columns:
    df_binary["fico_mid"] = (df_binary["fico_range_low"] + df_binary["fico_range_high"]) / 2
    bins = pd.cut(df_binary["fico_mid"], bins=15)
    fico_default = df_binary.groupby(bins)["bad_loan"].mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    fico_default.plot.bar(ax=ax, color="#e74c3c", edgecolor="white", alpha=0.85)
    ax.set_title("Default Rate by FICO Score Band", fontsize=14, fontweight="bold")
    ax.set_xlabel("FICO Midpoint Band")
    ax.set_ylabel("Default Rate")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 1.8 Key Takeaways

- **Class imbalance**: Default rate is approximately 18–22%. This is moderate; no extreme resampling needed.
- **FICO score** shows a strong monotonic inverse relationship with default — strong predictor.
- **Grade** (Lending Club's internal) is highly correlated with default rate.
- **DTI** and **revol_util** have right-skewed distributions — capping outliers will help.
- **Missing values**: `emp_length` and `revol_util` may have moderate missingness; imputation is needed.

Proceed to **02_feature_engineering.ipynb** →